In [1]:
import numpy as np
import pandas as pd
import networkx as nx

# ============================================================================
# MIR 元种群模型（精确 MMCA 实现）—— 核心函数完全不变
# ============================================================================

def generate_networks(N, avg_deg_info, avg_deg_mob, seed=42):
    np.random.seed(seed)
    p_info = avg_deg_info / (N-1) if N > 1 else 0
    G_info = nx.erdos_renyi_graph(N, p_info, seed=seed)
    Wv = nx.to_numpy_array(G_info, dtype=float)
    p_mob = avg_deg_mob / (N-1) if N > 1 else 0
    G_mob = nx.erdos_renyi_graph(N, p_mob, seed=seed+1)
    Wmob = nx.to_numpy_array(G_mob, dtype=float)
    R = np.zeros((N, N))
    for i in range(N):
        neighbors = np.where(Wmob[i] > 0)[0]
        deg = len(neighbors)
        if deg > 0:
            R[i, neighbors] = 1.0 / deg
        else:
            R[i, i] = 1.0
    return Wv, Wmob, R

def compute_flows(n, g, R):
    N = len(n)
    flow = np.zeros((N, N))
    for j in range(N):
        stay = (1 - g) * n[j]
        move_out = g * n[j]
        for i in range(N):
            if i == j:
                flow[j, i] = stay + move_out * R[j, i]
            else:
                flow[j, i] = move_out * R[j, i]
    return flow

def compute_patch_infection_risk(flow, pI, beta_U, beta_A):
    N = flow.shape[0]
    N_U = np.zeros(N)
    N_A = np.zeros(N)
    for i in range(N):
        prod_U = 1.0
        prod_A = 1.0
        for j in range(N):
            f = flow[j, i]
            if f > 0:
                prod_U *= (1 - beta_U * pI[j]) ** f
                prod_A *= (1 - beta_A * pI[j]) ** f
        N_U[i] = 1 - prod_U
        N_A[i] = 1 - prod_A
    return N_U, N_A

def compute_individual_infection_risk(N_U, N_A, g, R):
    Q_U = (1 - g) * N_U + g * np.dot(R, N_U)
    Q_A = (1 - g) * N_A + g * np.dot(R, N_A)
    return Q_U, Q_A

def compute_info_risk(states, Wv, lambda1):
    N = states.shape[0]
    pA = states[:, 1] + states[:, 2] + states[:, 4]
    r = np.zeros(N)
    for i in range(N):
        prod = 1.0
        for j in range(N):
            if Wv[j, i] > 0:
                prod *= (1 - lambda1 * pA[j])
        r[i] = 1 - prod
    return r

def update_states(states, r, Q_U, Q_A, delta, mu):
    US, AS, AI, UR, AR = states.T
    new_US = US * (1 - r) * (1 - Q_U) + AS * delta * (1 - Q_U)
    new_AS = AS * (1 - delta) * (1 - Q_A) + US * r * (1 - Q_A)
    new_AI = (AI * (1 - mu)
              + AS * (1 - delta) * Q_A
              + AS * delta * Q_U
              + US * (1 - r) * Q_U
              + US * r * Q_A)
    new_UR = AI * delta * mu + AR * delta + UR * (1 - r)
    new_AR = AR * (1 - delta) + AI * (1 - delta) * mu + UR * r
    new_states = np.vstack([new_US, new_AS, new_AI, new_UR, new_AR]).T
    row_sum = new_states.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1
    new_states /= row_sum
    return new_states

def main_simulation(Time, N, params):
    lambda1 = params["lambda1"]
    delta   = params["delta"]
    mu      = params["mu"]
    beta_U  = params["beta_U"]
    beta_A  = params["beta_A"]
    g       = params["g"]
    avg_deg_info = params["avg_deg_info"]
    avg_deg_mob  = params["avg_deg_mob"]
    seed    = params.get("seed", 42)

    Wv, Wmob, R = generate_networks(N, avg_deg_info, avg_deg_mob, seed)

    n = np.full(N, 200.0)
    states = np.zeros((N, 5))
    states[:, 0] = 0.99
    states[:, 2] = 0.01

    history = np.zeros((Time, 5))

    for t in range(Time):
        global_state = np.sum(states * n[:, None], axis=0) / np.sum(n)
        history[t] = global_state
        flow = compute_flows(n, g, R)
        pI = states[:, 2]
        N_U, N_A = compute_patch_infection_risk(flow, pI, beta_U, beta_A)
        Q_U, Q_A = compute_individual_infection_risk(N_U, N_A, g, R)
        r = compute_info_risk(states, Wv, lambda1)
        states = update_states(states, r, Q_U, Q_A, delta, mu)

    return history


# ============================================================================
# 扫描 λ 并计算峰值感染密度的重复实验
# ============================================================================
if __name__ == "__main__":
    Time = 150
    N = 15
    num_simulations = 50         # 每个 λ 重复实验次数
    base_seed = 12345             # 基础随机种子

    # 固定参数（λ 将被循环覆盖）
    base_params = {
        "lambda1": None,
        "delta": 0.15,
        "mu": 0.1,
        "beta_U": 0.003,
        "beta_A": 0.0015,
        "g": 0.2,
        "avg_deg_info": 2,
        "avg_deg_mob": 10,
    }

    # 扫描的 λ 值（与 CPS 模型保持一致）
    lambda_list = np.linspace(0.4, 0.8, 81)

    # 存储结果
    results = []

    for lambda1 in lambda_list:

        params = base_params.copy()
        params["lambda1"] = lambda1

        peak_list = []

        for sim in range(num_simulations):
            # 为每次实验生成独立且可重复的种子
            seed = base_seed + 1000
            params["seed"] = seed

            history = main_simulation(Time, N, params)
            ai_global = history[:, 2]          # AI 比例
            peak_infection = np.max(ai_global)
            peak_list.append(peak_infection)


        # 统计
        peak_mean = np.mean(peak_list)
        peak_std = np.std(peak_list, ddof=1)
        results.append({
            "lambda": lambda1,
            "peak_mean": peak_mean,
        })

    # 保存结果
    df_results = pd.DataFrame(results)
    df_results.to_csv("MIR_peak_vs_lambda.csv", index=False)
    print("\nMIR 模型 λ 扫描完成，结果已保存至 MIR_peak_vs_lambda.csv")
    print(df_results.head())


MIR 模型 λ 扫描完成，结果已保存至 MIR_peak_vs_lambda.csv
   lambda  peak_mean
0   0.400   0.375937
1   0.405   0.375488
2   0.410   0.375038
3   0.415   0.374588
4   0.420   0.374139


In [2]:
a = np.linspace(0.4, 0.8, 81)
a

array([0.4  , 0.405, 0.41 , 0.415, 0.42 , 0.425, 0.43 , 0.435, 0.44 ,
       0.445, 0.45 , 0.455, 0.46 , 0.465, 0.47 , 0.475, 0.48 , 0.485,
       0.49 , 0.495, 0.5  , 0.505, 0.51 , 0.515, 0.52 , 0.525, 0.53 ,
       0.535, 0.54 , 0.545, 0.55 , 0.555, 0.56 , 0.565, 0.57 , 0.575,
       0.58 , 0.585, 0.59 , 0.595, 0.6  , 0.605, 0.61 , 0.615, 0.62 ,
       0.625, 0.63 , 0.635, 0.64 , 0.645, 0.65 , 0.655, 0.66 , 0.665,
       0.67 , 0.675, 0.68 , 0.685, 0.69 , 0.695, 0.7  , 0.705, 0.71 ,
       0.715, 0.72 , 0.725, 0.73 , 0.735, 0.74 , 0.745, 0.75 , 0.755,
       0.76 , 0.765, 0.77 , 0.775, 0.78 , 0.785, 0.79 , 0.795, 0.8  ])